In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/RF/ANDROIDS"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["file_stem"] = df["file_stem"].astype(str).str.strip()
df["depressed"] = pd.to_numeric(df["depressed"], errors="coerce")

feature_cols = [f"mfcc_{i}" for i in range(1, 14)] + ["pitch_mean", "energy_mean"]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["depressed", "file_stem"]
df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols].astype(float).values
y = df_clean["depressed"].astype(int).values
groups = df_clean["file_stem"].values

print("Rows:", len(df_clean))
print("Groups:", df_clean["file_stem"].nunique())
print("Features used:", feature_cols)

# -----------------------------
# GroupKFold CV
# -----------------------------
gkf = GroupKFold(n_splits=5)

fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    })

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["file_stem"].nunique(),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "androids_rf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "androids_rf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

Rows: 224
Groups: 115
Features used: ['mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'pitch_mean', 'energy_mean']

Processing fold 1

Processing fold 2

Processing fold 3

Processing fold 4

Processing fold 5

Fold results:
   fold  n_train  n_test  accuracy        f1   roc_auc
0     1      179      45  0.733333  0.793103  0.814655
1     2      179      45  0.644444  0.680000  0.781746
2     3      179      45  0.533333  0.533333  0.736000
3     4      179      45  0.666667  0.727273  0.800000
4     5      180      44  0.681818  0.695652  0.714876

Summary:
  subset  n_rows  n_groups  accuracy_mean  accuracy_std   f1_mean   f1_std  \
0    all     224       115       0.651919      0.073923  0.685872  0.09567   

   roc_auc_mean  roc_auc_std  
0      0.769455     0.042501  

Results saved to:
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\RF\ANDROIDS


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/RF/RADAR"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["participant_id"] = df["participant_id"].astype(str).str.strip()
df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")

# Binary target
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

# RADAR feature columns
meta_cols = [
    "File", "participant_id", "Dataset", "Language", "Task",
    "recording_date", "Age", "Gender", "Education_Years",
    "Height", "phq8_score", "source_file", "depressed"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["depressed", "participant_id"]
df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols].astype(float).values
y = df_clean["depressed"].astype(int).values
groups = df_clean["participant_id"].values

print("Rows:", len(df_clean))
print("Groups:", df_clean["participant_id"].nunique())
print("Features used:", feature_cols)

# -----------------------------
# GroupKFold CV
# -----------------------------
gkf = GroupKFold(n_splits=5)

fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    })

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["participant_id"].nunique(),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "radar_rf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "radar_rf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

Rows: 8515
Groups: 274
Features used: ['Clip_Duration', 'Speaking_Rate', 'Articulation_Rate', 'Phonation_Ratio', 'Pause_Rate', 'Pause_Ratio', 'mean_F0', 'stdev_F0_Semitone', 'HNR_dB', 'Spectral_Slope', 'Spectral_Tilt', 'Cepstral_Peak_Prominence', 'mean_F1_Loc', 'std_F1_Loc', 'mean_B1_Loc', 'std_B1_Loc', 'mean_F2_Loc', 'std_F2_Loc', 'mean_B2_Loc', 'std_B2_Loc', 'Spectral_Gravity', 'Spectral_Std_Dev']

Processing fold 1

Processing fold 2

Processing fold 3

Processing fold 4

Processing fold 5

Fold results:
   fold  n_train  n_test  accuracy        f1   roc_auc
0     1     6812    1703  0.533764  0.330523  0.496456
1     2     6812    1703  0.525543  0.370717  0.513822
2     3     6812    1703  0.564885  0.379916  0.535093
3     4     6812    1703  0.602466  0.370233  0.553160
4     5     6812    1703  0.608925  0.427835  0.578167

Summary:
  subset  n_rows  n_groups  accuracy_mean  accuracy_std   f1_mean    f1_std  \
0    all    8515       274       0.567117      0.038221  0.375845  0

MERF for RADAR data


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

!pip install merf
!pip install sklearn.metrics
from sklearn.metrics import mean_squared_error

from merf.merf import MERF

# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/MERF/RADAR"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean data
# -----------------------------
df["recording_date"] = pd.to_datetime(df["recording_date"], errors="coerce")
df["participant_id"] = df["participant_id"].astype(str).str.strip()

# -----------------------------
# Outcome + grouping
# -----------------------------
y_score = df["phq8_score"]
ID_clusters = df["participant_id"]

# -----------------------------
# Random-effect covariates (Z)
# -----------------------------
z_candidates = [
    "Age",
    "Gender",
    "Education_Years",
    "Height"
]

z_features = [c for c in z_candidates if c in df.columns]
Z_factors = df[z_features].copy()

# -----------------------------
# Fixed-effect speech features (X)
# -----------------------------
x_candidates = [
    "Speaking_Rate",
    "Articulation_Rate",
    "Phonation_Ratio",
    "Pause_Rate",
    "Pause_Ratio",
    "mean_F0",
    "stdev_F0_Semitone",
    "HNR_dB",
    "Spectral_Slope",
    "Spectral_Tilt",
    "Cepstral_Peak_Prominence",
    "mean_F1_Loc",
    "std_F1_Loc",
    "mean_B1_Loc",
    "std_B1_Loc",
    "mean_F2_Loc",
    "std_F2_Loc",
    "mean_B2_Loc",
    "std_B2_Loc",
    "Spectral_Gravity",
    "Spectral_Std_Dev"
]

feature_cols = [c for c in x_candidates if c in df.columns]

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + z_features + ["phq8_score", "participant_id"]

df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols]
Z_factors = df_clean[z_features]
y_score = df_clean["phq8_score"]
ID_clusters = df_clean["participant_id"]

print("Rows:", len(df_clean))
print("Participants:", df_clean["participant_id"].nunique())
print("Features used:", feature_cols)
print("Random-effect covariates:", z_features)

# -----------------------------
# GroupKFold CV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rmse_list = []
fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y_score, groups=ID_clusters), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_score.iloc[train_index], y_score.iloc[test_index]

    clusters_train = ID_clusters.iloc[train_index]
    clusters_test = ID_clusters.iloc[test_index]

    Z_train = Z_factors.iloc[train_index]
    Z_test = Z_factors.iloc[test_index]

    # Scale X only
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # MERF model
    rf_reg = RandomForestRegressor(
        n_estimators=500,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    merf_model = MERF(
        fixed_effects_model=rf_reg,
        max_iterations=20
    )

    merf_model.fit(
        X_train_scaled,
        Z_train,
        clusters_train,
        y_train
    )

    y_pred = merf_model.predict(
        X_test_scaled,
        Z_test,
        clusters_test
    )

    rmse = mean_squared_error(
        y_test,
        y_pred,
    )

    rmse_list.append(rmse)

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "rmse": rmse
    })

    print(f"Fold RMSE = {rmse:.2f}")

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_participants": df_clean["participant_id"].nunique(),
    "rmse_mean": np.mean(rmse_list),
    "rmse_std": np.std(rmse_list)
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "radar_merf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "radar_merf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement sklearn.metrics (from versions: none)
ERROR: No matching distribution found for sklearn.metrics

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Rows: 8515
Participants: 274
Features used: ['Speaking_Rate', 'Articulation_Rate', 'Phonation_Ratio', 'Pause_Rate', 'Pause_Ratio', 'mean_F0', 'stdev_F0_Semitone', 'HNR_dB', 'Spectral_Slope', 'Spectral_Tilt', 'Cepstral_Peak_Prominence', 'mean_F1_Loc', 'std_F1_Loc', 'mean_B1_Loc', 'std_B1_Loc', 'mean_F2_Loc', 'std_F2_Loc', 'mean_B2_Loc', 'std_B2_Loc', 'Spectral_Gravity', 'Spectral_Std_Dev']
Random-effect covariates: ['Age', 'Gender', 'Education_Years', 'Height']

Processing fold 1


INFO     [merf.py:307] Training GLL is 8962.237606435323 at iteration 1.
INFO     [merf.py:307] Training GLL is 8332.936179210566 at iteration 2.
INFO     [merf.py:307] Training GLL is 7900.627229461784 at iteration 3.
INFO     [merf.py:307] Training GLL is 7546.638201721802 at iteration 4.
INFO     [merf.py:307] Training GLL is 7349.962171866001 at iteration 5.
INFO     [merf.py:307] Training GLL is 7170.5278410630735 at iteration 6.
INFO     [merf.py:307] Training GLL is 7066.501354912018 at iteration 7.
INFO     [merf.py:307] Training GLL is 6975.950192148225 at iteration 8.
INFO     [merf.py:307] Training GLL is 6927.95047483378 at iteration 9.
INFO     [merf.py:307] Training GLL is 6872.5152931853845 at iteration 10.
INFO     [merf.py:307] Training GLL is 6870.520065287191 at iteration 11.
INFO     [merf.py:307] Training GLL is 6843.6778514956995 at iteration 12.
INFO     [merf.py:307] Training GLL is 6844.7818948664035 at iteration 13.
INFO     [merf.py:307] Training GLL is 6809.

Fold RMSE = 41.85

Processing fold 2


INFO     [merf.py:307] Training GLL is 8826.42057455896 at iteration 1.
INFO     [merf.py:307] Training GLL is 8114.461499222596 at iteration 2.
INFO     [merf.py:307] Training GLL is 7635.592674310138 at iteration 3.
INFO     [merf.py:307] Training GLL is 7287.364551611366 at iteration 4.
INFO     [merf.py:307] Training GLL is 7058.40606341237 at iteration 5.
INFO     [merf.py:307] Training GLL is 6932.834043261982 at iteration 6.
INFO     [merf.py:307] Training GLL is 6827.642520353355 at iteration 7.
INFO     [merf.py:307] Training GLL is 6769.568954793412 at iteration 8.
INFO     [merf.py:307] Training GLL is 6688.142599277947 at iteration 9.
INFO     [merf.py:307] Training GLL is 6692.412418462437 at iteration 10.
INFO     [merf.py:307] Training GLL is 6656.5923126656935 at iteration 11.
INFO     [merf.py:307] Training GLL is 6635.2646336792495 at iteration 12.
INFO     [merf.py:307] Training GLL is 6640.206318384173 at iteration 13.
INFO     [merf.py:307] Training GLL is 6609.327

...

Fold RMSE = 37.56

Processing fold 4


INFO     [merf.py:307] Training GLL is 9040.491251682412 at iteration 1.
INFO     [merf.py:307] Training GLL is 8280.940869657376 at iteration 2.
INFO     [merf.py:307] Training GLL is 7764.759337164696 at iteration 3.
INFO     [merf.py:307] Training GLL is 7383.639131904722 at iteration 4.
INFO     [merf.py:307] Training GLL is 7139.949579602518 at iteration 5.
INFO     [merf.py:307] Training GLL is 7000.946981321617 at iteration 6.
INFO     [merf.py:307] Training GLL is 6887.225240387821 at iteration 7.
INFO     [merf.py:307] Training GLL is 6854.918794403609 at iteration 8.
INFO     [merf.py:307] Training GLL is 6786.74454576584 at iteration 9.
INFO     [merf.py:307] Training GLL is 6758.215853801831 at iteration 10.
INFO     [merf.py:307] Training GLL is 6754.700671787788 at iteration 11.
INFO     [merf.py:307] Training GLL is 6706.02761345433 at iteration 12.
INFO     [merf.py:307] Training GLL is 6692.276860308394 at iteration 13.
INFO     [merf.py:307] Training GLL is 6689.14718

Fold RMSE = 29.73

Processing fold 5


INFO     [merf.py:307] Training GLL is 8423.845258093043 at iteration 1.
INFO     [merf.py:307] Training GLL is 7570.821773096401 at iteration 2.
INFO     [merf.py:307] Training GLL is 7047.777057929546 at iteration 3.
INFO     [merf.py:307] Training GLL is 6678.407659965936 at iteration 4.
INFO     [merf.py:307] Training GLL is 6435.400299423183 at iteration 5.
INFO     [merf.py:307] Training GLL is 6232.601068449379 at iteration 6.
INFO     [merf.py:307] Training GLL is 6134.323531445293 at iteration 7.
INFO     [merf.py:307] Training GLL is 6064.25567855804 at iteration 8.
INFO     [merf.py:307] Training GLL is 6032.115564897553 at iteration 9.
INFO     [merf.py:307] Training GLL is 5999.269520376045 at iteration 10.
INFO     [merf.py:307] Training GLL is 5954.184209252365 at iteration 11.
INFO     [merf.py:307] Training GLL is 5897.227092346349 at iteration 12.
INFO     [merf.py:307] Training GLL is 5922.979294174617 at iteration 13.
INFO     [merf.py:307] Training GLL is 5890.8308

Fold RMSE = 35.73

Fold results:
   fold  n_train  n_test       rmse
0     1     6812    1703  41.854334
1     2     6812    1703  35.632446
2     3     6812    1703  37.555480
3     4     6812    1703  29.731719
4     5     6812    1703  35.730495

Summary:
  subset  n_rows  n_participants  rmse_mean  rmse_std
0    all    8515             274  36.100895  3.902331

Results saved to:
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\MERF\RADAR


Androids

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

from merf.merf import MERF

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/MERF/ANDROIDS"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["file_stem"] = df["file_stem"].astype(str).str.strip()
df["speech_type"] = df["speech_type"].astype(str).str.strip().str.lower()
df["subgroup_from_path"] = df["subgroup_from_path"].astype(str).str.strip().str.upper()
df["bdi_score"] = pd.to_numeric(df["bdi_score"], errors="coerce")

# -----------------------------
# Fixed-effect speech features (X)
# -----------------------------
feature_cols = [f"mfcc_{i}" for i in range(1, 14)] + ["pitch_mean", "energy_mean"]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Random-effect covariates (Z)
# -----------------------------
z_candidates = ["speech_type", "subgroup_from_path"]
z_features = [c for c in z_candidates if c in df.columns]

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["bdi_score", "file_stem"] + z_features
df_clean = df.dropna(subset=required_cols).copy()

# -----------------------------
# Build model inputs
# -----------------------------
X = df_clean[feature_cols].astype(float)
y_score = df_clean["bdi_score"].astype(float)
ID_clusters = df_clean["file_stem"]

Z_factors = pd.get_dummies(
    df_clean[z_features],
    drop_first=True
).astype(float)

print("Rows:", len(df_clean))
print("Unique groups:", df_clean["file_stem"].nunique())
print("Features used:", feature_cols)
print("Random-effect covariates:", z_features)

# -----------------------------
# GroupKFold CV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rmse_list = []
fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y_score, groups=ID_clusters), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y_score.iloc[train_index].astype(float)
    y_test = y_score.iloc[test_index].astype(float)

    clusters_train = ID_clusters.iloc[train_index]
    clusters_test = ID_clusters.iloc[test_index]

    Z_train = Z_factors.iloc[train_index].astype(float)
    Z_test = Z_factors.iloc[test_index].astype(float)

    # Scale X only
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # MERF model
    rf_reg = RandomForestRegressor(
        n_estimators=500,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    merf_model = MERF(
        fixed_effects_model=rf_reg,
        max_iterations=20
    )

    merf_model.fit(
        X_train_scaled,
        Z_train,
        clusters_train,
        y_train
    )

    y_pred = merf_model.predict(
        X_test_scaled,
        Z_test,
        clusters_test
    )

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    rmse_list.append(rmse)

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "rmse": rmse
    })

    print(f"Fold RMSE = {rmse:.2f}")

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["file_stem"].nunique(),
    "rmse_mean": np.mean(rmse_list),
    "rmse_std": np.std(rmse_list)
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "androids_merf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "androids_merf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

Rows: 209
Unique groups: 107
Features used: ['mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'pitch_mean', 'energy_mean']
Random-effect covariates: ['speech_type', 'subgroup_from_path']

Processing fold 1


INFO     [merf.py:307] Training GLL is 438.15914673862125 at iteration 1.
INFO     [merf.py:307] Training GLL is 567.9083580770808 at iteration 2.
INFO     [merf.py:307] Training GLL is 589.0985974427684 at iteration 3.
INFO     [merf.py:307] Training GLL is 572.4503116616996 at iteration 4.
INFO     [merf.py:307] Training GLL is 565.6081898565914 at iteration 5.
INFO     [merf.py:307] Training GLL is 575.4849182633338 at iteration 6.
INFO     [merf.py:307] Training GLL is 591.2834277168525 at iteration 7.
INFO     [merf.py:307] Training GLL is 606.1166959925696 at iteration 8.
INFO     [merf.py:307] Training GLL is 613.2490800715634 at iteration 9.
INFO     [merf.py:307] Training GLL is 612.5594401115243 at iteration 10.
INFO     [merf.py:307] Training GLL is 603.4308499960852 at iteration 11.
INFO     [merf.py:307] Training GLL is 600.6129207299232 at iteration 12.
INFO     [merf.py:307] Training GLL is 602.3082370381048 at iteration 13.
INFO     [merf.py:307] Training GLL is 598.762

Fold RMSE = 21.41

Processing fold 2


INFO     [merf.py:307] Training GLL is 371.15655400295844 at iteration 1.
INFO     [merf.py:307] Training GLL is 521.35240905953 at iteration 2.
INFO     [merf.py:307] Training GLL is 540.5040908255144 at iteration 3.
INFO     [merf.py:307] Training GLL is 546.2303035478773 at iteration 4.
INFO     [merf.py:307] Training GLL is 550.4709833690166 at iteration 5.
INFO     [merf.py:307] Training GLL is 565.4042968186764 at iteration 6.
INFO     [merf.py:307] Training GLL is 579.2536646919468 at iteration 7.
INFO     [merf.py:307] Training GLL is 597.4398355048991 at iteration 8.
INFO     [merf.py:307] Training GLL is 593.895846971493 at iteration 9.
INFO     [merf.py:307] Training GLL is 598.0641089931679 at iteration 10.
INFO     [merf.py:307] Training GLL is 604.7072649258893 at iteration 11.
INFO     [merf.py:307] Training GLL is 608.5397618727538 at iteration 12.
INFO     [merf.py:307] Training GLL is 607.1790730295334 at iteration 13.
INFO     [merf.py:307] Training GLL is 607.959385

Fold RMSE = 16.14

Processing fold 3


...

Fold RMSE = 10.47

Processing fold 4


INFO     [merf.py:307] Training GLL is 365.3115555315523 at iteration 1.
INFO     [merf.py:307] Training GLL is 521.5529700674005 at iteration 2.
INFO     [merf.py:307] Training GLL is 561.0994753054857 at iteration 3.
INFO     [merf.py:307] Training GLL is 573.7885471111726 at iteration 4.
INFO     [merf.py:307] Training GLL is 585.8212711329093 at iteration 5.
INFO     [merf.py:307] Training GLL is 598.225260071868 at iteration 6.
INFO     [merf.py:307] Training GLL is 617.9936993608966 at iteration 7.
INFO     [merf.py:307] Training GLL is 628.1602330428261 at iteration 8.
INFO     [merf.py:307] Training GLL is 648.3661564009033 at iteration 9.
INFO     [merf.py:307] Training GLL is 657.7902261917094 at iteration 10.
INFO     [merf.py:307] Training GLL is 665.9919391532478 at iteration 11.
INFO     [merf.py:307] Training GLL is 668.054296498467 at iteration 12.
INFO     [merf.py:307] Training GLL is 669.9472229845034 at iteration 13.
INFO     [merf.py:307] Training GLL is 671.618926

Fold RMSE = 14.45

Processing fold 5


INFO     [merf.py:307] Training GLL is 423.12097419589105 at iteration 1.
INFO     [merf.py:307] Training GLL is 577.7262128996961 at iteration 2.
INFO     [merf.py:307] Training GLL is 597.2476389161883 at iteration 3.
INFO     [merf.py:307] Training GLL is 594.8693954031938 at iteration 4.
INFO     [merf.py:307] Training GLL is 607.1292882199988 at iteration 5.
INFO     [merf.py:307] Training GLL is 612.4356172001286 at iteration 6.
INFO     [merf.py:307] Training GLL is 618.6907347842865 at iteration 7.
INFO     [merf.py:307] Training GLL is 633.1814793064746 at iteration 8.
INFO     [merf.py:307] Training GLL is 638.5012642399719 at iteration 9.
INFO     [merf.py:307] Training GLL is 648.7907355109447 at iteration 10.
INFO     [merf.py:307] Training GLL is 653.7013462305868 at iteration 11.
INFO     [merf.py:307] Training GLL is 657.7992190616675 at iteration 12.
INFO     [merf.py:307] Training GLL is 654.4508329878754 at iteration 13.
INFO     [merf.py:307] Training GLL is 650.541

Fold RMSE = 15.12

Fold results:
   fold  n_train  n_test       rmse
0     1      167      42  21.412788
1     2      167      42  16.141644
2     3      167      42  10.474043
3     4      167      42  14.445243
4     5      168      41  15.122853

Summary:
  subset  n_rows  n_groups  rmse_mean  rmse_std
0    all     209       107  15.519314  3.518122

Results saved to:
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\MERF\ANDROIDS
